# 支持向量机（SVM）（不展开核技巧）

模式识别：线性支持向量机——找**最大间隔**分界面。与 LDA 都在找分界，但 SVM 强调间隔与支持向量。

**范围：仅线性 SVM；不展开核技巧**（不讲核函数、RBF、核技巧推导）。


In [ ]:
import numpy as np
from sklearn.svm import LinearSVC, SVC
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=40, centers=2, random_state=42, cluster_std=1.2)
# 标签转为 ±1 便于手写间隔公式对照；sklearn 用 0/1 亦可
print("X shape", X.shape, "y unique", np.unique(y))


## 1. 线性 SVM 目标（直觉）

对可分数据，硬间隔 SVM 寻找法向量 \(w\) 与偏置 \(b\)，使两类间隔最大化，约束正确分类。  
软间隔（`C`）允许少量违例。本篇用 `LinearSVC` / 线性核 `SVC`，不涉及核映射。


In [ ]:
clf = LinearSVC(C=1.0, dual="auto", random_state=0)
clf.fit(X, y)
w, b = clf.coef_[0], clf.intercept_[0]
print("w =", w)
print("b =", b)

# 决策函数与预测
scores = X @ w + b
pred = (scores > 0).astype(int)
# LinearSVC 的类标签是 0/1，符号约定以 decision_function 为准
pred_sk = clf.predict(X)
print("训练准确率:", (pred_sk == y).mean())
print("decision_function 与 predict 一致?", np.array_equal((clf.decision_function(X) > 0).astype(int), pred_sk))


## 2. 与 LDA 的对照（一句话）

- **LDA**：假设类条件高斯等协方差，用散度矩阵求投影/分界。  
- **线性 SVM**：不显式建模类分布，直接优化间隔（及软间隔惩罚）。  

二者都能得到线性分界；数据与目标不同时边界不同。


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
lda.fit(X, y)
print("LDA 训练准确率:", (lda.predict(X) == y).mean())
print("SVM 训练准确率:", (clf.predict(X) == y).mean())
# 不要求系数相同，只说明都是线性分类器
print("SVM w:", np.round(w, 3))
print("LDA coef_:", np.round(lda.coef_[0], 3))


## 3. 范围声明

`SVC(kernel="linear")` 与 `LinearSVC` 同属线性 SVM。  
**本文不展开核技巧**：不讨论 `rbf` / `poly` 等核把数据映射到高维的做法。


In [ ]:
# 仅验证线性核 SVC 可跑通；不引入非线性核
svc_lin = SVC(kernel="linear", C=1.0)
svc_lin.fit(X, y)
print("SVC(linear) 准确率:", (svc_lin.predict(X) == y).mean())
print("支持向量个数:", len(svc_lin.support_))
